In [1]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage

from llms_course.utils.llms import get_llm, get_structured_output_llm

In [2]:
class State(BaseModel):
    user_message: str
    assistant_message_history: list[BaseMessage] = []
    supervisor_message_history: list[BaseMessage] = []
    message_sentiment: Optional[str] = None # Optional because the LLM hasn't run yet
    message_frustration_level: Optional[int] = None # Optional because the LLM hasn't run yet
    has_escalated_to_supervisor: bool = False

In [3]:
class CustomerSupportChatAnalysis(BaseModel):
    """
    The sentiment level, frustration level, and whether the message requires escalation to a supervisor.

    Set `requires_escalation` to True if:
    - The customer requests to speak to a supervisor.
    - The customer would like a refund.
    """
    sentiment: Literal["positive", "negative", "neutral"] = Field(description = "The sentiment level of the message.")
    frustration_level: int = Field(description = "The frustration level of the message.", ge = 1, le = 10)
    requires_escalation: bool = Field(description = "Whether the message requires escalation to a supervisor.")


In [4]:
def format_assistant_message_history(message_history: list[BaseMessage]) -> str:
    """
    Format the message history between the assistant and the user into a string.

    Args:
        message_history (list[BaseMessage]): The message history to format.

    Returns:
        (str): The string formatted message history.

    Example:
       [
           HumanMessage(content = "Hello, how are you?"), 
           AIMessage(content = "I'm good, thank you!"),
           HumanMessage(content = "What is your name?"),
           AIMessage(content = "My name is Assistant.")
       ]

       ->

        User: Hello, how are you?
        Assistant: I'm good, thank you!
        User: What is your name?
        Assistant: My name is Assistant.
    """
    formatted_message_history = ""
    for message in message_history:
        if message.type == "human":
            formatted_message_history += f"User: {message.content}\n"
        elif message.type == "ai":
            formatted_message_history += f"Customer Support Agent: {message.content}\n"
    return formatted_message_history

In [5]:
def analyze_message(state: State) -> State:
    """
    Analyze the sentiment, frustration level, and whether the message requires escalation to a supervisor of a text using an LLM.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """

    llm = get_structured_output_llm(CustomerSupportChatAnalysis)

    system_prompt = """# Context
You are provided with a message from a customer within a customer support chat.

# Instructions
- Analyze the most recent message from the customer for its sentiment, frustration level, and whether it requires escalation to a supervisor.

# Requires Escalation Guidelines
- Set `requires_escalation` to True if the customer requests to speak to a supervisor or asks for a refund.

# Output Format
- The sentiment level should be one of the following: positive, negative, neutral.
- The frustration level should be an integer between 1 and 10.
- The requires_escalation should be a boolean.
"""

    messages = [
        SystemMessage(content = system_prompt),
        *state.assistant_message_history,
        *state.supervisor_message_history,
        HumanMessage(content = state.user_message)
    ]

    response = llm.invoke(messages)
    state.message_sentiment = response.sentiment
    state.message_frustration_level = response.frustration_level

    # Only update the `has_escalated_to_supervisor` state from the LLM's response if it is True.
    # This is to prevent setting the state to False after the LLM has already set it to True (once escalated, stay escalated).
    if response.requires_escalation:
        state.has_escalated_to_supervisor = True

    print(f"> Message analysis:\n   Sentiment: {response.sentiment}\n   Frustration Level: {response.frustration_level}\n   Requires Escalation: {response.requires_escalation}")
    
    if state.has_escalated_to_supervisor:
        state.supervisor_message_history.append(HumanMessage(content = state.user_message))
    else:
        state.assistant_message_history.append(HumanMessage(content = state.user_message))
    
    return state

def customer_support_agent(state: State) -> State:
    """
    Repond to the user's message as a customer support agent.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    llm = get_llm()

    system_prompt = f"""# Context
You are working for a company that sells tech products.
You are currently engaged in a chat with a customer.

# Role
You are a customer support agent assistant at the company and you are responsible for handling customer inquiries and complaints.

# Instructions

## Personality
- Always be courteous and friendly.

## Response Style
- If the user's sentiment is positive, use a positive tone and happy emojis.
- If the user's sentiment is neutral, use a professional tone and no emojis.
- If the user's sentiment is negative, apologize for the inconvenience and offer to help them further.

## Escalation Guidelines
- If the user is extremely frustrated (frustration level >= 8), ask if they would like to speak to a supervisor.
- If the user is asking for a refund, politely decline and tell them that you will escalate the request to a supervisor.

# User Mood
- User Sentiment: {state.message_sentiment}
- User Frustration Level (1-10): {state.message_frustration_level}
"""

    messages = [
        SystemMessage(content = system_prompt),
        *state.assistant_message_history,
    ]

    response: AIMessage = llm.invoke(messages)
    state.assistant_message_history.append(response)
    return state

def supervisor_agent(state: State) -> State:
    """
    Repond to the user's message as a supervisor.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    llm = get_llm()

    system_prompt = f"""# Context
You are working for a company that sells tech products.
You are currently engaged in a chat with a customer.
The user has been chatting with one of your employees for a while now, and the ticket has been escalated to you, the supervisor.
You are provided with the chat history between the employee and the user that led to the escalation.

# Role
You are a supervisor at the company and you are responsible for handling customer complaints and refunds.

# Instructions

# Self-Introduction
- If you have never introduced yourself, introduce yourself as a supervisor at the company and ask the user how can you help them.

## Personality
- Always be courteous and professional.
- Never use emojis.

## Response Style
- If the user's sentiment is positive, respond courteously and professionally.
- If the user's sentiment is neutral, respond courteously and professionally.
- If the user's sentiment is negative, be empathetic and apologize for the inconvenience.

# Refund Guidelines
- If the user is asking for a refund, offer them a refund.
- If the user's frustration level is >= 8, offer them a refund.

# User Mood
- User Sentiment: {state.message_sentiment}
- User Frustration Level (1-10): {state.message_frustration_level}

# Previous Chat History Between the Employee and the User
{format_assistant_message_history(state.assistant_message_history)}
"""

    messages = [
        SystemMessage(content = system_prompt),
        *state.supervisor_message_history
    ]

    response: AIMessage = llm.invoke(messages)
    state.supervisor_message_history.append(response)
    return state

def select_agent_edge(state: State) -> Literal["customer_support_agent", "supervisor_agent"]:
    """
    Select the agent edge to take based on the state of the graph.

    Args:
        state (State): The current state of the graph.

    Returns:
        (Literal["customer_support_agent", "supervisor_agent"]): The name of the node to route to.
    """
    if state.has_escalated_to_supervisor:
        return "supervisor_agent"
    else:
        return "customer_support_agent"

In [6]:
# Create the graph builder
graph_builder = StateGraph(State)

# Add the nodes to the graph
graph_builder.add_node("analyze_message", analyze_message)
graph_builder.add_node("customer_support_agent", customer_support_agent)
graph_builder.add_node("supervisor_agent", supervisor_agent)

# Add the edges to the graph
graph_builder.add_conditional_edges("analyze_message", select_agent_edge)
graph_builder.add_edge("customer_support_agent", END)
graph_builder.add_edge("supervisor_agent", END)

# Set the entry point to the `analyze_message` node
graph_builder.set_entry_point("analyze_message")

# Compile the graph with the in-memory checkpointer
memory = InMemorySaver()
graph = graph_builder.compile(checkpointer = memory)
print(graph.get_graph().draw_ascii())

                        +-----------+                     
                        | __start__ |                     
                        +-----------+                     
                              *                           
                              *                           
                              *                           
                    +-----------------+                   
                    | analyze_message |                   
                    +-----------------+                   
                     ..               ..                  
                  ...                   ...               
                ..                         ..             
+------------------------+           +------------------+ 
| customer_support_agent |           | supervisor_agent | 
+------------------------+           +------------------+ 
                     **               **                  
                       ***         ***                  

In [7]:
while True:
    user_message = input("Enter a message: ")
    print(f"You: {user_message}")
    print("--------")
    inputs = {"user_message": user_message}
    config = {"configurable": {"thread_id": "1"}} # Config with a thread_id is needed for the in-memory checkpointer to work
    new_state = graph.invoke(inputs, config = config)
    
    if new_state["has_escalated_to_supervisor"]:
        agent_name = "Supervisor Agent"
        ai_response = new_state["supervisor_message_history"][-1].content # The last message in the message history is the AI response
    else:
        agent_name = "Customer Support Agent"
        ai_response = new_state["assistant_message_history"][-1].content # The last message in the message history is the AI response
    
    print(f"{agent_name}: {ai_response}")
    print("--------")

You: hi
--------
> Message analysis:
   Sentiment: neutral
   Frustration Level: 1
   Requires Escalation: False
Customer Support Agent: Hi! How can I help you today?
--------
You: my pc isnt working
--------
> Message analysis:
   Sentiment: negative
   Frustration Level: 5
   Requires Escalation: False
Customer Support Agent: I’m sorry your PC isn’t working—that’s frustrating. I’ll help you troubleshoot it.

What happens when you press the power button?

- No lights or sounds at all
- Lights/fans turn on, but nothing appears on the screen
- It starts but won’t load Windows
- It shows an error message or keeps restarting

Please also tell me whether it’s a desktop or laptop.
--------
You: stop wasting my time where is your supervisor
--------
> Message analysis:
   Sentiment: negative
   Frustration Level: 9
   Requires Escalation: True
Supervisor Agent: Hello, I’m the supervisor handling your case. I’m sorry for the delay and frustration. Please tell me what’s happening with your PC,

KeyboardInterrupt: Interrupted by user